In [ ]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt

# Define the model
model = pybamm.BaseModel()

# Define the variables
x = pybamm.Variable("x")  # anode
y = pybamm.Variable("y")  # cathode
s = pybamm.Variable("s")  # SEI
e = pybamm.Variable("e")  # electrolyte
z = pybamm.Variable("z")  # z
T = pybamm.Variable("T")  # temperature

# Define constants (ensure realistic values)
A_x = 2.5 * 10**13
A_y = 6.67 * 10**11
A_s = 1.67 * 10**15
A_e = 3.37 * 10**12
E_s = 3.24 * 10**(-19)
E_x = 2.24 * 10**(-19)
E_y = 2.03 * 10**(-19)
E_e = 1.58* 10**(-19)
z_0 = 0.033
T_0 = 12 + 273.15  # Converted to Kelvin
Q_heater = 25
C = 2.4
V=4.2
n=0.28
h=200
A=100000
E=0.9
M_a= 8.1 * 10**(-3)
M_c= 18.3 * 10**(-3)
h_a=1714
h_c=314
h_s=257
S=1
rho=2865.5
C_p=830



k_b = 1.38e-23  # Boltzmann constant in J/K

# Safeguard for potential numerical issues with very small/large values
exp1 = pybamm.exp(-E_x / (k_b * T))
exp2 = pybamm.exp(-z / z_0)
exp3 = pybamm.exp(-E_y / (k_b * T))
exp4 = pybamm.exp(-E_s / (k_b * T))
exp5 = pybamm.exp(-E_e / (k_b * T))
exp6 = (T*4) - (T_0*4)
exp7 = T - T_0

# Ensure exp1 and exp2 are within valid numerical ranges
#exp1 = pybamm.minimum(pybamm.maximum(exp1, 1e-10), 1e10)
#exp2 = pybamm.minimum(pybamm.maximum(exp2, 1e-10), 1e10)

# Define the right-hand side (RHS) of the ODE
dxdt = -A_x * x * exp1 * exp2
dydt = A_y * y * (1 - y) * exp3
dsdt = -A_s * s * exp4
dedt = A_e * exp5
dzdt = A_x * x * exp1 * exp2
dTdt =( Q_heater -(A * E * exp6) -(A * h * exp7) + ( M_a * h_a * A_x * x * exp1 * exp2 ) + ( M_c * h_c * A_y * y * (1-y) * exp3 ) + ( M_a * h_s * A_s * s * exp4 ) - (  3600 * C * V * A_e * S * exp5 * n   ) )/ ( rho * C_p) 

# Set the RHS of the ODE in the model
model.rhs = {x: dxdt, y: dydt, z: dzdt, s: dsdt, e: dedt, T: dTdt}

# Define the initial conditions
model.initial_conditions = {
    x: pybamm.Scalar(0.75),
    y: pybamm.Scalar(1),
    z: pybamm.Scalar(1.75),
    s: pybamm.Scalar(2.75),
    e: pybamm.Scalar(3.75),
    T: pybamm.Scalar(300)  # initial temperature
}

# Add model variables for output
model.variables = {
    "x": x,
    "y": y,
    "z": z,
    "s": s,
    "e": e,
    "T": T  # Include T for temperature output
}

# Discretize the model
disc = pybamm.Discretisation()  # Use default discretisation
disc.process_model(model)

# Solve the model
solver = pybamm.ScipySolver()
t = np.linspace(0, 50, 100)  # Time range
solution = solver.solve(model, t)

# Extract the solution for variables
x_sol = solution["x"]
y_sol = solution["y"]
z_sol = solution["z"]
s_sol = solution["s"]
e_sol = solution["e"]
T_sol = solution["T"]
t_sol = solution.t  # Time points

# Plot the results for variable x
#plt.plot(t_sol, x_sol(t_sol), label="x")
#plt.xlabel("Time (s)")
#plt.ylabel("x")
#plt.legend()
#plt.title("Decomposition of Anode (x) over time")
#plt.show()

# Plot the results for temperature T
plt.plot(t_sol, T_sol(t_sol), label="T")
plt.xlabel("Time (s)")
plt.ylabel("Temperature (K)")
plt.legend()
plt.title("Temperature (T) over time")
plt.show()